In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

def plot_individual_error_learning_curves(data):
    data = data.copy()

    for sub in data["subid"].unique():
        sub_mask = data["subid"] == sub
        max_trial_block2 = data.loc[sub_mask & (data["Block"] == 2), "Trial"].max()
        data.loc[sub_mask & (data["Block"] == 3), "Trial"] += max_trial_block2

    subject_colours = np.load("../figures/subject_colours.npy")
    plt.figure(figsize=(14, 5))
    wide_data = data.groupby(["Trial", "subid"])["Error"].mean().unstack()

    for i, sub in enumerate(wide_data.columns):
        plt.plot(wide_data.index, wide_data[sub], color=subject_colours[i], linewidth=1, alpha=0.75)

    baseline_start = data[data["Block"] == 2]["Trial"].min()
    baseline_end = data[data["Block"] == 2]["Trial"].max()
    rotation_start = baseline_end
    rotation_end = data["Trial"].max()
    baseline_length = baseline_end - baseline_start

    plt.axvspan(baseline_start, baseline_end, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(rotation_start, rotation_start + baseline_length, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(rotation_end - baseline_length, rotation_end, color="lightgrey", alpha=0.3, zorder=0)

    plt.axhline(0, color="black", linestyle=":", alpha=1, zorder=5)
    plt.axvline(baseline_end, color="black", linestyle="--", alpha=1, zorder=5)

    plt.ylabel("Error", fontsize=16)
    plt.yticks(fontsize=14)
    plt.xticks([baseline_end / 2, (baseline_end + rotation_end) / 2], ["Baseline", "Learning"], fontsize=16)
    plt.title("Individual Motor Learning Curves", fontsize=20)

    ax = plt.gca()
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.savefig("../figures/behaviour/individual_motor_learning_curves.png", dpi=800, transparent=True)
    plt.close()

def plot_individual_reinforcement_learning_curves(data):
    data = data.copy()
    data = data[data["Block"].isin([1, 2]) & data["Valid"].astype(bool)]

    for sub in data["subid"].unique():
        sub_mask = data["subid"] == sub
        max_trial_block1 = data.loc[sub_mask & (data["Block"] == 1), "Trial"].max()
        data.loc[sub_mask & (data["Block"] == 2), "Trial"] += max_trial_block1

    subject_colours = np.load("../figures/subject_colours.npy")

    plt.figure(figsize=(11, 4))
    wide = data.groupby(["Trial", "subid"])["Score"].mean().unstack()
    for i, sub in enumerate(wide.columns):
        plt.plot(wide.index, wide[sub], color=subject_colours[i], linewidth=1, alpha=0.75)

    block1_trials = np.sort(data.loc[data["Block"] == 1, "Trial"].unique())
    block2_trials = np.sort(data.loc[data["Block"] == 2, "Trial"].unique())

    block1_start, block1_end = block1_trials[0], block1_trials[-1]
    block2_start, block2_end = block2_trials[0], block2_trials[-1]

    n_highlight = 64

    last64_b1_start = block1_trials[-n_highlight] if len(block1_trials) >= n_highlight else block1_start
    first64_b2_end = block2_trials[n_highlight - 1] if len(block2_trials) >= n_highlight else block2_end
    last64_b2_start = block2_trials[-n_highlight] if len(block2_trials) >= n_highlight else block2_start

    plt.axvspan(last64_b1_start, block1_end, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(block2_start, first64_b2_end, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(last64_b2_start, block2_end, color="lightgrey", alpha=0.3, zorder=0)

    plt.axvline(70, color="black", linestyle="--", alpha=1, zorder=5, linewidth=1)
    plt.axhline(0, color="black", linestyle=":", alpha=1, zorder=5, linewidth=1)

    plt.ylabel("Score", fontsize=16)
    plt.yticks(fontsize=14)
    plt.xticks([block1_end / 2, (block1_end + block2_end) / 2], ["Baseline", "Learning"], fontsize=16)
    plt.title("Individual Reinforcement Learning Curves", fontsize=20)

    ax = plt.gca()
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.savefig("../figures/behaviour/individual_reinforcement_learning_curves.png", dpi=800, transparent=True)
    plt.close()

def plot_error_learning(data):
    data = data.copy()

    for sub in data['subid'].unique():
        sub_mask = data["subid"] == sub
        max_trial_block2 = data[(sub_mask) & (data['Block'] == 2)]['Trial'].max()
        data.loc[(sub_mask) & (data['Block'] == 3), 'Trial'] += max_trial_block2

    plt.figure(figsize=(14,5))

    wide_data   = data.groupby(["Trial", "subid"])["Error"].mean().unstack()
    group_error = wide_data.mean(axis=1)
    group_sem   = wide_data.sem(axis=1)
    
    plt.plot(group_error.index, group_error.values, color="black", linewidth=1)
    plt.fill_between(group_error.index, group_error.values - group_sem.values, group_error.values + group_sem.values, color="grey", alpha=0.25, zorder=1)

    baseline_start = data[data["Block"] == 2]["Trial"].min()
    baseline_end   = data[data["Block"] == 2]["Trial"].max()
    rotation_start = baseline_end
    rotation_end   = data["Trial"].max()
    baseline_length = baseline_end - baseline_start

    plt.axvspan(baseline_start, baseline_end, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(rotation_start, rotation_start+baseline_length, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(rotation_end - baseline_length, rotation_end, color="lightgrey", alpha=0.3, zorder=0)
    plt.axhline(0, color='black', linestyle=':', alpha=1, zorder=5)
    
    plt.axvline(baseline_end, color="black", linestyle="--", alpha=1, zorder=5)

    plt.ylabel("Error", fontsize=16)
    plt.yticks(fontsize=14)
    plt.xticks([baseline_end / 2, (baseline_end + rotation_end) / 2],["Baseline", "Learning"],fontsize=16)
    plt.title("Motor Learning Curves", fontsize=20)
    plt.tight_layout()
    plt.xticks(group_error.index)
    ax = plt.gca()
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.savefig("../figures/behaviour/motor_learning.png", dpi=800, transparent=True)
    plt.close()

def plot_indiv_error_learning(data):
    df = (data[(data["subid"] == "sub-19") & (data["Block"] == 3)].sort_values("Trial").reset_index(drop=True))
    
    early = df.head(64).dropna(subset=["Error"])
    late  = df.tail(64).dropna(subset=["Error"])

    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(7.6,7.6))
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.grid(False)
    ax.spines["polar"].set_visible(False)
    ax.axvline(0, color="black", linestyle="-", linewidth=3, alpha=0.9, zorder=10)
    ax.axvline(np.deg2rad(45), color="black", linestyle="-", linewidth=3, alpha=0.9, zorder=10)
    ax.set_yticklabels([])

    fixed_radius = 1
    theta_early = np.radians(early["Error"].values)
    theta_late  = np.radians(late["Error"].values)

    cmap = plt.cm.Purples
    
    colours_early = cmap(np.linspace(0.2, 0.6, len(early)))
    colours_late  = cmap(np.linspace(0.6, 1.0, len(late)))

    for i, theta in enumerate(theta_early):
        ax.plot([theta, theta], [0, fixed_radius], color=colours_early[i], linewidth=1, alpha=0.8)
    for i, theta in enumerate(theta_late):
        ax.plot([theta, theta], [0, fixed_radius], color=colours_late[i],  linewidth=1, alpha=0.8)
    plt.savefig("../figures/behaviour/indiv_motor.png", dpi=800, transparent=True)
    plt.close()

    fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=(6, 2), constrained_layout=True)
    cmap_early = plt.cm.Purples
    norm_early = matplotlib.colors.Normalize(vmin=1, vmax=64)
    cb1 = matplotlib.colorbar.ColorbarBase(ax1, cmap=cmap_early, norm=norm_early, orientation='horizontal')
    cb1.set_label('Early learning: Trial number')
    cb1.set_ticks([1, 64])
    cb1.set_ticklabels(['Trial 1', 'Trial 64'])
    
    cmap_late = plt.cm.Purples
    norm_late = matplotlib.colors.Normalize(vmin=1, vmax=64)
    cb2 = matplotlib.colorbar.ColorbarBase(ax2, cmap=cmap_late, norm=norm_late, orientation='horizontal')
    cb2.set_label('Late learning: Trial number')
    cb2.set_ticks([1, 64])
    cb2.set_ticklabels(['Trial 1', 'Trial 64'])
    
    plt.savefig("../figures/behaviour/indiv_motor_cmap.png", dpi=800, transparent=True)
    plt.close()

def plot_error_learning_summary(summary, subject_colours):
    group_means = [summary["early_score"].mean(), summary["late_score"].mean()]
    group_sems = [summary["early_score"].std() / np.sqrt(len(summary)),
                  summary["late_score"].std() / np.sqrt(len(summary))]
    
    fig, ax = plt.subplots(figsize=(2, 6))
    plt.bar([0, 1], group_means, yerr=group_sems, color=["lightgrey", "lightgrey"],
            alpha=0.6, capsize=5, width=0.8, zorder=0)

    np.random.seed(0)
    for i, (pid, row) in enumerate(summary.iterrows()):
        jitter = np.random.uniform(-0.05, 0.05, 2)
        xs = np.array([0, 1]) + jitter
        ys = [row["early_score"], row["late_score"]]

        plt.scatter(xs, ys, color=subject_colours[i], s=80, zorder=3, edgecolor="black", linewidth=0.5)
        plt.plot(xs, ys, color=subject_colours[i], alpha=0.7, linewidth=2, zorder=2)

    plt.xticks([0, 1], ["Early", "Late"], fontsize=14)
    plt.ylabel("Adaptation", fontsize=14)
    plt.title("Motor Learning: Early vs Late Adaptation", fontsize=16)
    for spine in ["top", "right"]:
        plt.gca().spines[spine].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    plt.tight_layout()
    plt.savefig("../figures/behaviour/motor_learning_early_late.png", dpi=800, transparent=True)
    plt.close()

def error_adaptation_scores(data):
    def compute_scores(g):
        early = g.iloc[:64]["Error"].mean() 
        late = g.iloc[-64:]["Error"].mean() 
        learning = late - early
        return pd.Series({"early_score": early, "late_score": late, "learning": learning})
    
    data = data.loc[data["Block"] == 3].copy()
    summary = (data.groupby("subid", as_index=False).apply(compute_scores, include_groups=False)
              .reset_index(drop=True))

    rows = []
    for pid, g in data.groupby("subid"):
        early = (g[g["Trial"] <= 64][["Trial", "Error"]].set_index("Trial").reindex(range(1, 65))
                 .rename(columns={"Error": "early_score"}).reset_index(drop=True)) 
        late = (g[g["Trial"] > 160 - 64][["Trial", "Error"]].set_index("Trial").reindex(range(160 - 63, 160 + 1))
                .rename(columns={"Error": "late_score"}).reset_index(drop=True)) 
        paired = pd.concat([early, late], axis=1)  
        paired["subid"] = pid
        paired["learning"] = paired["late_score"] - paired["early_score"]
        rows.append(paired[["subid", "early_score", "late_score", "learning"]])

    trial_data = pd.concat(rows, ignore_index=True)

    subject_colors = np.load("../figures/subject_colours.npy")
    plot_error_learning_summary(summary, subject_colors)
    return trial_data, summary  

def plot_reinforcement_learning(data):
    data = data[data["Block"].isin([1, 2]) & data["Valid"].astype(bool)]
    
    for subject in data["subid"].unique():
        subj_mask = data["subid"] == subject
        max_trial_block1 = data[(subj_mask) & (data["Block"] == 1)]["Trial"].max()
        data.loc[(subj_mask) & (data["Block"] == 2), "Trial"] += max_trial_block1

    plt.figure(figsize=(11,4))
    wide = data.groupby(["Trial", "subid"])["Score"].mean().unstack()
    group_score = wide.mean(axis=1)
    group_sem   = wide.sem(axis=1)
    plt.plot(group_score.index, group_score.values, color="black", linewidth=1, label="Group mean")
    plt.fill_between(group_score.index, group_score.values - group_sem.values, group_score.values + group_sem.values,
                     color="grey", alpha=0.25, zorder=1)

    block1_trials = np.sort(data.loc[data["Block"] == 1, "Trial"].unique())
    block2_trials = np.sort(data.loc[data["Block"] == 2, "Trial"].unique())

    block1_start, block1_end = block1_trials[0], block1_trials[-1]
    block2_start, block2_end = block2_trials[0], block2_trials[-1]

    n_highlight = 64
    last64_b1_start = block1_trials[-n_highlight] if len(block1_trials) >= n_highlight else block1_start
    plt.axvspan(last64_b1_start, block1_end, color="lightgrey", alpha=0.3, zorder=0)

    first64_b2_end = block2_trials[n_highlight-1] if len(block2_trials) >= n_highlight else block2_end
    plt.axvspan(block2_start, first64_b2_end, color="lightgrey", alpha=0.3, zorder=0)

    last64_b2_start = block2_trials[-n_highlight] if len(block2_trials) >= n_highlight else block2_start
    plt.axvspan(last64_b2_start, block2_end, color="lightgrey", alpha=0.3, zorder=0)

    plt.axvline(70, color="black", linestyle="--", alpha=1, zorder=5, linewidth=1)
    plt.axhline(0, color='black', linestyle=':', alpha=1, zorder=5, linewidth=1)
    plt.ylabel("Score", fontsize=16)
    plt.yticks(fontsize=14)
    plt.xticks([block1_end / 2, (block1_end + block2_end) / 2], ["Baseline", "Learning"], fontsize=16)
    plt.title("Reinforcement Learning Curves", fontsize=20)
    plt.tight_layout()
    ax = plt.gca()
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.xticks(group_score.index)
    plt.savefig("../figures/behaviour/reinforcement_learning.png", dpi=800, transparent=True)
    plt.close()

def plot_indiv_reinforcement_learning(data):
    df = (data[(data["subid"] == "sub-15") & (data["Block"] == 2)].sort_values("Trial").reset_index(drop=True))

    early = df.head(64).dropna(subset=["LateralMove"])["LateralMove"].tolist()
    late = df.tail(64).dropna(subset=["LateralMove"])["LateralMove"].tolist()

    plt.figure(figsize=(4, 6))
    plt.plot([0.25, 0.75], [0, 0], c="red", linewidth=2) 
    plt.scatter(0.5, -0.5, s=200, c='black', zorder=3)    

    x_start, y_start = 0.5, -0.5
    x_end, y_end = 0.5, 0
    t = np.linspace(0, 1, 100)
    y_curve = y_start + (y_end - y_start) * t
    
    plt.plot(0.5 + 0.1 * np.sin(np.pi * t), y_curve, c='black', linewidth=2, zorder=10)
    plt.plot(0.5 - 0.1 * np.sin(np.pi * t), y_curve, c='black', linestyle="--", linewidth=2, zorder=10)

    cmap = plt.cm.Purples
    
    cmap_early = cmap(np.linspace(0.2, 0.6, len(early)))
    cmap_late  = cmap(np.linspace(0.6, 1.0, len(late)))
    
    for i, lm in enumerate(early):
        offset = (lm / 4.0) * 0.1
        x_curve = 0.5 + offset * np.sin(np.pi * t)
        plt.plot(x_curve, y_curve, color=cmap_early[i], linewidth=1, alpha=0.7)
    
    for i, lm in enumerate(late):
        offset = (lm / 4.0) * 0.1
        x_curve = 0.5 + offset * np.sin(np.pi * t)
        plt.plot(x_curve, y_curve, color=cmap_late[i], linewidth=1, alpha=0.7)
    
    plt.xlim(0.1, 0.9)
    plt.ylim(-0.6, 0.05)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.savefig("../figures/behaviour/indiv_reinforcement.png", dpi=800, transparent=True)
    plt.close()

def plot_reinforcement_learning_summary(summary, subject_colours):
    group_means = [summary["early_score"].mean(), summary["late_score"].mean()]
    group_sems = [summary["early_score"].std() / np.sqrt(len(summary)),
                  summary["late_score"].std() / np.sqrt(len(summary))]

    fig, ax = plt.subplots(figsize=(2, 6))
    plt.bar([0, 1], group_means, yerr=group_sems, color=["lightgrey", "lightgrey"],
            alpha=0.6, capsize=5, width=0.8, zorder=0)

    np.random.seed(0)
    for i, (pid, row) in enumerate(summary.iterrows()):
        jitter = np.random.uniform(-0.05, 0.05, 2)
        xs = np.array([0, 1]) + jitter
        ys = [row["early_score"], row["late_score"]]

        plt.scatter(xs, ys, color=subject_colours[i], s=80, zorder=3, edgecolor="black", linewidth=0.5)
        plt.plot(xs, ys, color=subject_colours[i], alpha=0.7, linewidth=2, zorder=2)

    plt.xticks([0, 1], ["Early", "Late"], fontsize=14)
    plt.ylabel("Score", fontsize=14)
    plt.title("Reinforcement Learning: Early vs Late Score", fontsize=16)

    for spine in ["top", "right"]:
        plt.gca().spines[spine].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    plt.tight_layout()
    plt.savefig("../figures/behaviour/reinforcement_learning_early_late.png", dpi=800, transparent=True)
    plt.close()
    
def reinforcement_learning_scores(data):
    def early_late_scores(g: pd.DataFrame) -> pd.Series:
        early_mean = g.iloc[:64]["Score"].mean()
        late_mean  = g.iloc[-64:]["Score"].mean()   
        learning   = late_mean - early_mean
        return pd.Series({"early_score": early_mean, "late_score": late_mean, "learning": learning})

    data = data.loc[data["Block"] == 2].copy()
    summary = (data.groupby("subid", as_index=False).apply(early_late_scores, include_groups=False)
              .reset_index(drop=True))

    rows = []
    for pid, g in data.groupby("subid"):
        early = (g[g["Trial"] <= 64][["Trial", "Score"]].set_index("Trial").reindex(range(1, 65))
                 .rename(columns={"Score": "early_score"}).reset_index(drop=True))  
        late = (g[g["Trial"] > 201 - 64][["Trial", "Score"]].set_index("Trial").reindex(range(201 - 63, 201 + 1))
                .rename(columns={"Score": "late_score"}).reset_index(drop=True))  
        paired = pd.concat([early, late], axis=1) 
        paired["subid"] = pid
        paired["learning"] = paired["late_score"] - paired["early_score"]
        rows.append(paired[["subid", "early_score", "late_score", "learning"]])
    
    trial_data = pd.concat(rows, ignore_index=True)
    subject_colours = np.load("../figures/subject_colours.npy")
    plot_reinforcement_learning_summary(summary, subject_colours)
    return trial_data, summary  

def _plot_error_learning(data):
    data = data.copy()

    for sub in data['subid'].unique():
        sub_mask = data["subid"] == sub
        max_trial_block2 = data[(sub_mask) & (data['Block'] == 2)]['Trial'].max()
        data.loc[(sub_mask) & (data['Block'] == 3), 'Trial'] += max_trial_block2

    plt.figure(figsize=(14, 5))

    wide_data   = data.groupby(["Trial", "subid"])["Error"].mean().unstack()
    group_error = wide_data.mean(axis=1)
    group_sem   = wide_data.sem(axis=1)

    for sub in wide_data.columns:
        plt.plot(wide_data.index, wide_data[sub], color="grey", linewidth=0.75, alpha=0.5, zorder=2)

    plt.fill_between(group_error.index, group_error.values - group_sem.values,
                     group_error.values + group_sem.values, color="grey", alpha=0.4, zorder=3,)
    plt.plot(group_error.index, group_error.values, color="black", linewidth=1.5, zorder=4)

    baseline_start = data[data["Block"] == 2]["Trial"].min()
    baseline_end   = data[data["Block"] == 2]["Trial"].max()
    rotation_start = baseline_end
    rotation_end   = data["Trial"].max()
    baseline_length = baseline_end - baseline_start

    plt.axvspan(baseline_start, baseline_end, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(rotation_start, rotation_start + baseline_length, color="lightgrey", alpha=0.3, zorder=0)
    plt.axvspan(rotation_end - baseline_length, rotation_end, color="lightgrey", alpha=0.3, zorder=0)
    plt.axhline(0, color='black', linestyle=':', alpha=1, zorder=5)
    plt.axvline(baseline_end, color="black", linestyle="--", alpha=1, zorder=5)

    plt.ylabel("Error", fontsize=16)
    plt.yticks(fontsize=14)
    plt.xticks([baseline_end / 2, (baseline_end + rotation_end) / 2], ["Baseline", "Learning"], fontsize=16)
    plt.title("Motor Learning Curves", fontsize=20)
    y_min = (group_error - group_sem).min()
    y_max = (group_error + group_sem).max()
    y_pad = (y_max - y_min) * 0.05
    plt.ylim(y_min - y_pad, y_max + y_pad)
    plt.tight_layout()
    plt.xticks(group_error.index)
    ax = plt.gca()
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.savefig("../figures/behaviour/motor_learning.png", dpi=800, transparent=True)
    plt.close()

def _plot_reinforcement_learning(data):
    data = data[data["Block"].isin([1, 2]) & data["Valid"].astype(bool)]

    for subject in data["subid"].unique():
        subj_mask = data["subid"] == subject
        max_trial_block1 = data[(subj_mask) & (data["Block"] == 1)]["Trial"].max()
        data.loc[(subj_mask) & (data["Block"] == 2), "Trial"] += max_trial_block1

    plt.figure(figsize=(11, 4))
    wide        = data.groupby(["Trial", "subid"])["Score"].mean().unstack()
    group_score = wide.mean(axis=1)
    group_sem   = wide.sem(axis=1)

    for sub in wide.columns:
        plt.plot(wide.index, wide[sub], color="grey", linewidth=0.75, alpha=0.5, zorder=2)

    plt.fill_between(group_score.index, group_score.values - group_sem.values,
                     group_score.values + group_sem.values, color="grey", alpha=0.4, zorder=3,)
    plt.plot(group_score.index, group_score.values, color="black", linewidth=1.5, label="Group mean", zorder=4)

    block1_trials = np.sort(data.loc[data["Block"] == 1, "Trial"].unique())
    block2_trials = np.sort(data.loc[data["Block"] == 2, "Trial"].unique())

    block1_start, block1_end = block1_trials[0], block1_trials[-1]
    block2_start, block2_end = block2_trials[0], block2_trials[-1]

    n_highlight = 64
    last64_b1_start = block1_trials[-n_highlight] if len(block1_trials) >= n_highlight else block1_start
    plt.axvspan(last64_b1_start, block1_end, color="lightgrey", alpha=0.3, zorder=0)

    first64_b2_end = block2_trials[n_highlight - 1] if len(block2_trials) >= n_highlight else block2_end
    plt.axvspan(block2_start, first64_b2_end, color="lightgrey", alpha=0.3, zorder=0)

    last64_b2_start = block2_trials[-n_highlight] if len(block2_trials) >= n_highlight else block2_start
    plt.axvspan(last64_b2_start, block2_end, color="lightgrey", alpha=0.3, zorder=0)

    plt.axvline(70, color="black", linestyle="--", alpha=1, zorder=5, linewidth=1)
    plt.axhline(0, color='black', linestyle=':', alpha=1, zorder=5, linewidth=1)
    plt.ylabel("Score", fontsize=16)
    plt.yticks(fontsize=14)
    plt.xticks([block1_end / 2, (block1_end + block2_end) / 2], ["Baseline", "Learning"], fontsize=16)
    plt.title("Reinforcement Learning Curves", fontsize=20)
    # For plot_reinforcement_learning:
    y_min = (group_score - group_sem).min()
    y_max = (group_score + group_sem).max()
    y_pad = (y_max - y_min) * 0.05
    plt.ylim(0, y_max + y_pad)

    plt.tight_layout()
    ax = plt.gca()
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.xticks(group_score.index)
    plt.savefig("../figures/behaviour/reinforcement_learning.png", dpi=800, transparent=True)
    plt.close()
    
def main():
    cmap = matplotlib.colormaps.get_cmap("turbo")
    subject_colours = cmap(np.linspace(0, 1, 37))   
    np.save("../figures/subject_colours.npy", subject_colours)

    os.makedirs("../figures/behaviour", exist_ok=True)

    subject_list = pd.read_csv("../data/behavioural-data/participants.tsv", sep="\t")[lambda df: df["exclude"] == 0]
    subject_list["participant_id"] = "sub-" + subject_list["participant_id"].astype(str).str.zfill(2)
    behavioural_ids = subject_list["dicom_dir"].to_list()
    subid_map = dict(zip(subject_list["dicom_dir"], subject_list["participant_id"]))

    error_learning = pd.read_csv("../data/behavioural-data/ml_trial_data.csv")
    error_learning["Error"] = np.degrees(error_learning["Error"])
    error_learning = error_learning[error_learning["Subject"].isin(behavioural_ids)]
    error_learning["subid"] = error_learning["Subject"].map(subid_map)
    error_learning = error_learning[error_learning['Block'].isin([2, 3])]
    error_learning = error_learning[["subid", "Block", "Trial", "Error"]]
    error_learning.loc[error_learning["Block"] == 3, "Error"] = 45 - error_learning.loc[error_learning["Block"] == 3, "Error"]

    _plot_error_learning(error_learning)
    plot_indiv_error_learning(error_learning)
    sub_adaptation, group_adaptation = error_adaptation_scores(error_learning)
    group_adaptation.to_csv("../output/group_adaptation.csv")

    reinforcement_learning = pd.read_csv("../data/behavioural-data/rl_trial_data.csv")
    reinforcement_learning = reinforcement_learning[reinforcement_learning["Subject"].isin(behavioural_ids)]
    reinforcement_learning["subid"] = reinforcement_learning["Subject"].map(subid_map)
    
    _plot_reinforcement_learning(reinforcement_learning)
    plot_indiv_reinforcement_learning(reinforcement_learning)
    sub_reinforcement, group_reinforcement = reinforcement_learning_scores(reinforcement_learning)
    group_reinforcement.to_csv("../output/group_reinforcement.csv")

main()